# Predicting Short-Term Stock Market Movements Using Financial News Sentiment Analysis

**CIS 635 – Data Mining and Knowledge Discovery – Fall 2026**
**Team:** Collins Kemboi; [second team member, if applicable]

This project investigates whether sentiment from financial news can predict short-term stock movements. We gather historical news headlines and daily stock data, use natural language processing (FinBERT) to score the sentiment of each headline, and apply data mining techniques: logistic regression, decision trees, random forests, support vector machines, and k-means clustering.

**Research question:** Does news sentiment improve next-day directional prediction beyond historical price and volume features?

The notebook follows the structure of the final report:

1. Introduction and problem motivation
2. Data description
3. Data preprocessing and integration
4. Methodology
5. Results
6. Discussion and limitations
7. Conclusion and future work
8. References and dataset citations

**How to run:** pick a `DATA_SOURCE` in the settings cell below, then *Runtime → Run all*. A GPU runtime (*Runtime → Change runtime type → T4 GPU*) makes the FinBERT step much faster.

## 0. Setup

In [ ]:
# transformers and torch come preinstalled on Colab; this only installs them if they're missing.
import importlib.util
if importlib.util.find_spec("transformers") is None:
    %pip install -q transformers torch

In [ ]:
import io
import re
import time
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, silhouette_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

In [ ]:
# ---------------- Settings ----------------
# "online"    -> download news from GDELT and prices from Stooq (slow the first time, then cached)
# "upload"    -> use CSV files you put in data/raw/ (market_<TICKER>.csv and news_<TICKER>.csv)
# "synthetic" -> made-up data, only for checking that the notebook runs end to end
DATA_SOURCE = "synthetic"

TICKERS = ["AAPL", "MSFT", "AMZN", "TSLA", "NVDA"]
START_DATE = "2022-01-01"
END_DATE = "2024-12-31"

USE_FINBERT = True          # False = simple word-list scorer (only for quick tests)
STOOQ_API_KEY = None        # only needed if Stooq asks for one
RANDOM_STATE = 42

RAW_DIR = Path("data/raw")
OUTPUT_DIR = Path("outputs")
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Introduction and problem motivation

News about a company (earnings, lawsuits, product launches, analyst upgrades) is one of the main ways new information reaches investors. If markets don't absorb that information instantly, the tone of today's headlines might tell us something about tomorrow's price move. On the other hand, a lot of news is already priced in by the time it is published, so its extra predictive value is an open question.

We test this directly. We build two versions of every model:

* **Market only:** features from past prices and trading volume.
* **Market + sentiment:** the same features plus daily FinBERT sentiment of the company's news headlines.

If the second version does consistently better on unseen data, news sentiment adds information beyond what is already in prices. We predict **direction** (up or down tomorrow), not the exact price, and this is not a trading system.

## 2. Data description

| | Dataset 1: Financial news | Dataset 2: Stock market data |
|---|---|---|
| Source | GDELT DOC 2.0 API (gdeltproject.org) | Stooq (stooq.com) |
| One row is | a headline (title, URL, website, publish time in UTC) | one trading day for one stock (open, high, low, close, volume) |
| Coverage | 2017 onward, English-language web news | daily history for U.S. listed stocks |
| Terms | free for research/academic use with citation | free download; check terms before use |

**How they relate:** both datasets share the **stock ticker** and **time**. We map every headline to the trading session it can affect (Section 3.3), add up the sentiment of all headlines per stock per session, and join that onto the price table using *ticker + date*. The price data gives us the label (did the stock go up the next day?), and the news gives us the sentiment features, so the analysis needs both.

Stocks studied: Apple, Microsoft, Amazon, Tesla and Nvidia. These are large companies with a lot of daily news coverage.

### 2.1 Download functions

GDELT returns at most 250 articles per request and asks for about one request every 5 seconds, so we download a few days at a time and save the results to CSV files. Running this a second time reads the saved files instead of downloading again.

In [ ]:
COMPANIES = {
    # query = what we search in GDELT; pattern = what a headline must contain to count
    "AAPL": {"name": "Apple",     "query": '("Apple Inc" OR "Apple" OR "iPhone" OR AAPL)', "pattern": r"\b(?:apple|iphone|aapl|tim cook)\b"},
    "MSFT": {"name": "Microsoft", "query": '("Microsoft" OR MSFT)', "pattern": r"\b(?:microsoft|msft|satya nadella)\b"},
    "AMZN": {"name": "Amazon",    "query": '("Amazon" OR AMZN)',    "pattern": r"\b(?:amazon|amzn|aws|andy jassy)\b"},
    "TSLA": {"name": "Tesla",     "query": '("Tesla" OR TSLA)',     "pattern": r"\b(?:tesla|tsla|elon musk)\b"},
    "NVDA": {"name": "Nvidia",    "query": '("Nvidia" OR NVDA)',    "pattern": r"\b(?:nvidia|nvda|jensen huang)\b"},
}


def download_prices_from_stooq(ticker, start, end, api_key=None):
    params = {"s": ticker.lower() + ".us", "i": "d",
              "d1": pd.Timestamp(start).strftime("%Y%m%d"), "d2": pd.Timestamp(end).strftime("%Y%m%d")}
    if api_key:
        params["apikey"] = api_key
    response = requests.get("https://stooq.com/q/d/l/", params=params, timeout=30)
    response.raise_for_status()
    if not response.text.lower().startswith("date"):
        raise RuntimeError(f"Stooq didn't return a CSV for {ticker}: {response.text[:200]}")
    return pd.read_csv(io.StringIO(response.text))


def request_gdelt(query, start, end):
    params = {"query": query + " sourcelang:english", "mode": "artlist", "format": "json",
              "maxrecords": 250, "sort": "datedesc",
              "startdatetime": start.strftime("%Y%m%d%H%M%S"), "enddatetime": end.strftime("%Y%m%d%H%M%S")}
    response = requests.get("https://api.gdeltproject.org/api/v2/doc/doc", params=params, timeout=60)
    response.raise_for_status()
    if not response.text.strip().startswith("{"):  # errors come back as plain text
        raise RuntimeError("GDELT error: " + response.text[:200])
    return response.json().get("articles", [])


def download_headlines_from_gdelt(ticker, start, end, days_per_request=3, wait_seconds=5.5):
    articles = []
    window_start = pd.Timestamp(start)
    last_day = pd.Timestamp(end) + pd.Timedelta(days=1)
    while window_start < last_day:
        window_end = min(window_start + pd.Timedelta(days=days_per_request), last_day)
        for attempt in range(3):  # retry a couple of times if GDELT is busy
            try:
                articles += request_gdelt(COMPANIES[ticker]["query"], window_start, window_end)
                break
            except (requests.RequestException, RuntimeError, ValueError):
                if attempt == 2:
                    raise
                time.sleep(wait_seconds * (attempt + 2))
        window_start = window_end
        time.sleep(wait_seconds)
    if not articles:
        return pd.DataFrame(columns=["published_utc", "headline", "url", "source"])
    raw = pd.DataFrame(articles)
    return pd.DataFrame({
        "published_utc": pd.to_datetime(raw["seendate"], format="%Y%m%dT%H%M%SZ", utc=True),
        "headline": raw["title"], "url": raw["url"], "source": raw["domain"],
    })

### 2.2 Synthetic data (for testing only)

When `DATA_SOURCE = "synthetic"`, we generate fake prices and headlines with a small, built-in link between today's news tone and tomorrow's return. This lets us check that every step of the notebook runs, and that the models can find a signal when one exists. **Numbers from synthetic data must not be reported as findings.**

In [ ]:
def make_synthetic_data(tickers, start, end, signal=0.006, seed=0):
    rng = np.random.default_rng(seed)
    days = pd.bdate_range(start, end)
    days = days[[(d.month, d.day) not in [(1, 1), (7, 4), (12, 25)] for d in days]]  # a few holidays
    templates = {
        "pos": ["{} shares surge after earnings beat", "{} gains on strong growth outlook", "Analysts upgrade {} on record profit"],
        "neg": ["{} shares plunge after earnings miss", "{} falls on weak outlook", "Analysts downgrade {} amid probe"],
        "neu": ["{} to hold annual shareholder meeting", "{} names new board member", "What to watch for {} this week"],
    }
    prices, news = {}, {}
    for ticker in tickers:
        name = COMPANIES[ticker]["name"]
        tone = rng.normal(0, 1, len(days))
        returns = rng.normal(0.0003, 0.018, len(days))
        returns[1:] += signal * tone[:-1]          # yesterday's news nudges today's return
        close = 100 * np.exp(np.cumsum(returns))
        open_ = close * np.exp(rng.normal(0, 0.005, len(days)))
        prices[ticker] = pd.DataFrame({
            "Date": days, "Open": open_, "Close": close,
            "High": np.maximum(open_, close) * (1 + abs(rng.normal(0, 0.008, len(days)))),
            "Low": np.minimum(open_, close) * (1 - abs(rng.normal(0, 0.008, len(days)))),
            "Volume": (rng.lognormal(16, 0.3, len(days)) * (1 + 0.3 * abs(tone))).round(),
        })
        rows = []
        previous_close = (days[0] - pd.Timedelta(hours=8)).tz_localize("America/New_York")
        for day, t in zip(days, tone):
            close_time = (day + pd.Timedelta(hours=16)).tz_localize("America/New_York")
            for _ in range(rng.poisson(3 * (1 + abs(t)))):
                kind = "neu" if rng.random() < 0.35 else ("pos" if rng.random() < 1 / (1 + np.exp(-1.5 * t)) else "neg")
                published = previous_close + (close_time - previous_close) * rng.uniform(0, 0.999)
                rows.append({"published_utc": published.tz_convert("UTC"),
                             "headline": rng.choice(templates[kind]).format(name), "url": None, "source": "synthetic"})
            previous_close = close_time
        news[ticker] = pd.DataFrame(rows)
        news[ticker] = pd.concat([news[ticker], news[ticker].sample(frac=0.05, random_state=seed)])  # fake reposts
    return prices, news

### 2.3 Load both datasets

In [ ]:
raw_prices, raw_news = {}, {}

if DATA_SOURCE == "synthetic":
    raw_prices, raw_news = make_synthetic_data(TICKERS, START_DATE, END_DATE, seed=RANDOM_STATE)
else:
    for ticker in TICKERS:
        price_file = RAW_DIR / f"market_{ticker}.csv"
        news_file = RAW_DIR / f"news_{ticker}.csv"

        if not price_file.exists():
            if DATA_SOURCE == "upload":
                raise FileNotFoundError(f"Please upload {price_file}")
            download_prices_from_stooq(ticker, START_DATE, END_DATE, STOOQ_API_KEY).to_csv(price_file, index=False)

        if not news_file.exists():
            if DATA_SOURCE == "upload":
                raise FileNotFoundError(f"Please upload {news_file}")
            print(f"Downloading {ticker} headlines from GDELT (this takes a while)...")
            download_headlines_from_gdelt(ticker, START_DATE, END_DATE).to_csv(news_file, index=False)

        raw_prices[ticker] = pd.read_csv(price_file)
        raw_news[ticker] = pd.read_csv(news_file)

size_table = pd.DataFrame({
    "price rows": {t: len(raw_prices[t]) for t in TICKERS},
    "headlines (raw)": {t: len(raw_news[t]) for t in TICKERS},
})
size_table

## 3. Data preprocessing and integration

### 3.1 Cleaning the price data
We standardize column names, remove repeated dates and days with a missing or non-positive closing price, and keep only the date range we study.

In [ ]:
def clean_prices(prices, ticker):
    prices = prices.rename(columns=lambda c: c.strip().lower())
    prices = prices[["date", "open", "high", "low", "close", "volume"]].copy()
    prices["date"] = pd.to_datetime(prices["date"]).dt.normalize()
    for col in ["open", "high", "low", "close", "volume"]:
        prices[col] = pd.to_numeric(prices[col], errors="coerce")
    prices = prices.dropna(subset=["close"]).drop_duplicates(subset="date", keep="last")
    prices = prices[(prices["close"] > 0) & (prices["volume"].fillna(0) >= 0)]
    prices = prices[(prices["date"] >= START_DATE) & (prices["date"] <= END_DATE)]
    prices.insert(0, "ticker", ticker)
    return prices.sort_values("date").reset_index(drop=True)


prices = pd.concat([clean_prices(raw_prices[t], t) for t in TICKERS], ignore_index=True)
prices.groupby("ticker")["date"].agg(["min", "max", "count"])

### 3.2 Cleaning the headlines
Three problems we need to deal with:

1. **Missing values**: headlines without a title or timestamp are dropped.
2. **Ambiguous matches**: a search for "Apple" also finds recipes. We keep a headline only if it matches the company's name pattern.
3. **Duplicates**: the same story is often reposted by many websites. We compare headlines in lowercase without punctuation and keep only the first copy per day.

In [ ]:
def clean_headlines(news, ticker):
    news = news.copy()
    news["published_utc"] = pd.to_datetime(news["published_utc"], utc=True)
    news["headline"] = news["headline"].astype("string").str.strip()
    before = len(news)

    news = news.dropna(subset=["published_utc", "headline"])
    news = news[news["headline"].str.len() > 0]
    missing_removed = before - len(news)

    about_company = news["headline"].str.contains(COMPANIES[ticker]["pattern"], flags=re.IGNORECASE, regex=True)
    off_topic_removed = (~about_company).sum()
    news = news[about_company]

    news["simple_text"] = news["headline"].str.lower().str.replace(r"[^a-z0-9 ]", "", regex=True).str.split().str.join(" ")
    news["day"] = news["published_utc"].dt.floor("D")
    n = len(news)
    news = news.sort_values("published_utc").drop_duplicates(subset=["simple_text", "day"])
    duplicates_removed = n - len(news)

    news = news.drop(columns=["simple_text", "day"])
    news.insert(0, "ticker", ticker)
    report = {"raw": before, "missing": missing_removed, "off-topic": off_topic_removed,
              "duplicates": duplicates_removed, "kept": len(news)}
    return news.reset_index(drop=True), report


cleaned, cleaning_report = [], {}
for ticker in TICKERS:
    news_clean, cleaning_report[ticker] = clean_headlines(raw_news[ticker], ticker)
    cleaned.append(news_clean)
headlines = pd.concat(cleaned, ignore_index=True)

pd.DataFrame(cleaning_report).T   # how many rows each cleaning step removed

### 3.3 Matching headlines to trading sessions (the join key)

This is the most important step for avoiding **look-ahead bias** (accidentally using information from the future).

We pretend to make each prediction at the market close (4:00 pm New York time) of day *t*, and predict whether the stock closes higher on day *t+1*. So:

* a headline published **before 4:00 pm** on a trading day belongs to **that** day;
* a headline published **after 4:00 pm**, or on a **weekend or holiday**, belongs to the **next** trading day.

GDELT timestamps are in UTC, so we convert to New York time first (this also handles daylight saving time). The list of trading days comes from the price data, so weekends and holidays are automatically skipped.

In [ ]:
def match_to_trading_day(news, prices):
    matched = []
    for ticker, group in news.groupby("ticker"):
        trading_days = np.sort(prices.loc[prices["ticker"] == ticker, "date"].unique())
        ny_time = group["published_utc"].dt.tz_convert("America/New_York")
        day = ny_time.dt.tz_localize(None).dt.normalize()
        earliest = day.where(ny_time.dt.hour < 16, day + pd.Timedelta(days=1))  # after close -> next day
        position = np.searchsorted(trading_days, earliest.to_numpy(dtype="datetime64[ns]"))
        ok = position < len(trading_days)          # news after the last trading day is dropped
        group = group[ok].copy()
        group["session"] = pd.to_datetime(trading_days[position[ok]])
        matched.append(group)
    return pd.concat(matched, ignore_index=True)


headlines = match_to_trading_day(headlines, prices)

# Quick check: show a few headlines with their New York time and assigned session
check = headlines.sample(8, random_state=1)[["ticker", "published_utc", "session", "headline"]].copy()
check["new_york_time"] = check["published_utc"].dt.tz_convert("America/New_York").dt.strftime("%a %Y-%m-%d %H:%M")
check[["ticker", "new_york_time", "session", "headline"]]

### 3.4 Scoring sentiment with FinBERT

FinBERT (`ProsusAI/finbert`) is a BERT language model fine-tuned on financial text. For each headline it gives the probability of being **positive**, **negative**, or **neutral**. We use it instead of a general-purpose sentiment tool because words like "liability" or "volatile" mean something different in finance.

If FinBERT can't be loaded, the notebook falls back to a simple word-counting scorer so it still runs, but that fallback is only for testing.

In [ ]:
LABELS = ["positive", "negative", "neutral"]


def finbert_scores(texts, batch_size=32):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device).eval()
    # Find which output column is which label
    model_labels = {i: label.lower() for i, label in model.config.id2label.items()}
    order = [next(i for i, l in model_labels.items() if l == wanted) for wanted in LABELS]

    results = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = tokenizer(texts[i:i + batch_size], padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            probs = torch.softmax(model(**batch).logits, dim=-1).cpu().numpy()
            results.append(probs[:, order])
            if (i // batch_size) % 50 == 0:
                print(f"  scored {i + len(batch['input_ids']):,} / {len(texts):,}")
    return np.vstack(results)


POSITIVE_WORDS = {"beat", "beats", "surge", "surges", "gain", "gains", "jump", "record", "growth", "upgrade",
                  "strong", "profit", "rise", "rises", "rally", "boost", "raises", "higher", "tops"}
NEGATIVE_WORDS = {"miss", "misses", "plunge", "plunges", "fall", "falls", "drop", "slump", "loss", "downgrade",
                  "weak", "lawsuit", "probe", "recall", "cut", "cuts", "decline", "layoffs", "warns", "lower"}


def word_list_scores(texts):
    scores = []
    for text in texts:
        words = re.findall(r"[a-z]+", text.lower())
        raw = np.array([sum(w in POSITIVE_WORDS for w in words), sum(w in NEGATIVE_WORDS for w in words), 0.5])
        scores.append(np.exp(raw) / np.exp(raw).sum())
    return np.array(scores)


texts = headlines["headline"].astype(str).tolist()
scorer_used = "word list (fallback)"
if USE_FINBERT:
    try:
        probs = finbert_scores(texts)
        scorer_used = "FinBERT"
    except Exception as error:
        print("Couldn't load FinBERT, using the word-list scorer instead:", error)
        probs = word_list_scores(texts)
else:
    probs = word_list_scores(texts)

headlines["p_pos"], headlines["p_neg"], headlines["p_neu"] = probs[:, 0], probs[:, 1], probs[:, 2]
headlines["sentiment"] = [LABELS[i] for i in probs.argmax(axis=1)]
print("Scorer used:", scorer_used)
headlines["sentiment"].value_counts(normalize=True)

In [ ]:
# Examples of the most positive and most negative headlines - a sanity check on the scores
headlines["net_sentiment"] = headlines["p_pos"] - headlines["p_neg"]
examples = pd.concat([headlines.nlargest(5, "net_sentiment"), headlines.nsmallest(5, "net_sentiment")])
examples[["ticker", "headline", "p_pos", "p_neg", "p_neu"]]

### 3.5 Daily sentiment and joining the two datasets

We combine all headlines for the same stock and trading session into one row (average probabilities, share of positive/negative headlines, number of headlines) and then **left-join** that onto the price table on *ticker + date*. A left join keeps trading days with no news; for those days the counts and sentiment are set to 0 and a `has_news` flag is set to 0.

In [ ]:
headlines["is_pos"] = (headlines["sentiment"] == "positive").astype(float)
headlines["is_neg"] = (headlines["sentiment"] == "negative").astype(float)

daily_sentiment = headlines.groupby(["ticker", "session"]).agg(
    news_count=("headline", "size"),
    sent_pos_mean=("p_pos", "mean"),
    sent_neg_mean=("p_neg", "mean"),
    sent_neu_mean=("p_neu", "mean"),
    sent_net_mean=("net_sentiment", "mean"),
    sent_net_std=("net_sentiment", "std"),
    sent_pos_share=("is_pos", "mean"),
    sent_neg_share=("is_neg", "mean"),
).reset_index().rename(columns={"session": "date"})
daily_sentiment["sent_net_std"] = daily_sentiment["sent_net_std"].fillna(0)

data = prices.merge(daily_sentiment, on=["ticker", "date"], how="left")
sentiment_columns = [c for c in daily_sentiment.columns if c not in ("ticker", "date")]
data[sentiment_columns] = data[sentiment_columns].fillna(0)
print(f"{len(data):,} stock-days after the join; {(data['news_count'] > 0).mean():.1%} have at least one headline")
data.head()

### 3.6 Feature engineering and the label

**Label:** `target = 1` if tomorrow's close is higher than today's, else 0.

**Market features** (all computed only from past and current data):

| Feature | Meaning |
|---|---|
| `ret_1` … `ret_5` | log return today, yesterday, … 4 days ago |
| `ret_5d_cum` | total return over the last 5 days |
| `vol_5`, `vol_20` | volatility: standard deviation of returns over 5 and 20 days |
| `volume_chg` | change in log volume from yesterday |
| `volume_z20` | how unusual today's volume is compared to the last 20 days |
| `hl_range` | (high − low) / close, a measure of intraday movement |
| `close_sma10_ratio` | how far the close is above/below its 10-day average |

**Sentiment features:** the daily averages from 3.5, plus `has_news`, log of the headline count, yesterday's net sentiment, and 3-day rolling averages.

In [ ]:
MARKET_FEATURES = ["ret_1", "ret_2", "ret_3", "ret_4", "ret_5", "ret_5d_cum", "vol_5", "vol_20",
                   "volume_chg", "volume_z20", "hl_range", "close_sma10_ratio"]
SENTIMENT_FEATURES = ["has_news", "news_count_log", "sent_pos_mean", "sent_neg_mean", "sent_neu_mean",
                      "sent_net_mean", "sent_net_std", "sent_pos_share", "sent_neg_share",
                      "sent_net_mean_lag1", "sent_net_3d", "news_count_3d_log"]


def add_features(stock):
    stock = stock.sort_values("date").copy()
    r = np.log(stock["close"]).diff()
    for k in range(1, 6):
        stock[f"ret_{k}"] = r.shift(k - 1)
    stock["ret_5d_cum"] = r.rolling(5).sum()
    stock["vol_5"] = r.rolling(5).std()
    stock["vol_20"] = r.rolling(20).std()
    log_volume = np.log1p(stock["volume"])
    stock["volume_chg"] = log_volume.diff()
    stock["volume_z20"] = (log_volume - log_volume.rolling(20).mean()) / log_volume.rolling(20).std()
    stock["hl_range"] = (stock["high"] - stock["low"]) / stock["close"]
    stock["close_sma10_ratio"] = stock["close"] / stock["close"].rolling(10).mean() - 1

    stock["has_news"] = (stock["news_count"] > 0).astype(float)
    stock["news_count_log"] = np.log1p(stock["news_count"])
    stock["sent_net_mean_lag1"] = stock["sent_net_mean"].shift(1).fillna(0)
    stock["sent_net_3d"] = stock["sent_net_mean"].rolling(3, min_periods=1).mean()
    stock["news_count_3d_log"] = np.log1p(stock["news_count"].rolling(3, min_periods=1).sum())

    # Label: does the stock close higher tomorrow? (the last day has no tomorrow -> NaN)
    stock["next_close"] = stock["close"].shift(-1)
    stock["target"] = np.where(stock["next_close"] > stock["close"], 1.0, np.nan)
    stock.loc[stock["next_close"] <= stock["close"], "target"] = 0.0
    return stock


data = pd.concat([add_features(stock) for _, stock in data.groupby("ticker")], ignore_index=True)
data = data.dropna(subset=MARKET_FEATURES + ["target"])   # first ~20 days lack history; last day lacks a label
data["target"] = data["target"].astype(int)
data = data.sort_values(["date", "ticker"]).reset_index(drop=True)
print(f"Modeling table: {len(data):,} rows, {len(MARKET_FEATURES)} market + {len(SENTIMENT_FEATURES)} sentiment features")

### 3.7 Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
data.groupby("ticker")["news_count"].mean().plot.bar(ax=axes[0], title="Average headlines per trading day")
data.groupby("ticker")["target"].mean().plot.bar(ax=axes[1], title="Share of up days (class balance)")
axes[1].axhline(0.5, ls="--", c="grey")
news_days = data[data["has_news"] == 1].copy()
news_days["next_return"] = np.log(news_days["next_close"] / news_days["close"])
quintile = pd.qcut(news_days["sent_net_mean"], 5, labels=["most negative", "2", "3", "4", "most positive"], duplicates="drop")
news_days.groupby(quintile, observed=True)["next_return"].mean().plot.bar(ax=axes[2], title="Next-day return by sentiment quintile")
axes[2].tick_params(axis="x", labelrotation=20)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda.png", dpi=130)
plt.show()

In [ ]:
# Correlation of each sentiment feature with the next-day label
data[SENTIMENT_FEATURES + ["target"]].corr()["target"].drop("target").sort_values().plot.barh(
    figsize=(6, 4), title="Correlation with next-day up/down label")
plt.tight_layout()
plt.show()

## 4. Methodology

**Chronological split.** Stock data is a time series, so we never shuffle. The first 60% of trading days are used for training, the next 20% for validation and the last 20% for the final test. We leave a one-day gap between periods because the label of the last training day uses the next day's close.

**Models.**

* **Logistic regression:** simple linear baseline, easy to interpret.
* **Decision tree:** captures simple rules and interactions (e.g. "negative news *and* high volatility").
* **Random forest:** many trees averaged together; usually more stable than one tree.
* **Support vector machine (RBF kernel):** can find non-linear boundaries.
* **Majority-class baseline:** always predicts the most common direction. A model must beat this to be useful.

**Tuning.** Each model has a small grid of settings. We train every setting on the training period and pick the one with the best **macro F1** on the validation period. We use macro F1 instead of accuracy so that a model that predicts "up" every day doesn't win by default. `class_weight="balanced"` is one of the options to deal with class imbalance. We then retrain the chosen setting on train + validation and evaluate it **once** on the test period.

**Preventing leakage.** Features only use data up to day *t*; the scaler sits inside a scikit-learn `Pipeline`, so it only learns from the training rows; and the test period is never used for any choice.

**Evaluation.** Accuracy, balanced accuracy, precision, recall, F1, ROC-AUC and confusion matrices, for market-only vs market + sentiment.

In [ ]:
dates = np.sort(data["date"].unique())
n = len(dates)
train_end, val_end = int(n * 0.6), int(n * 0.8)
GAP = 1

train = data[data["date"].isin(dates[:train_end - GAP])]
val = data[data["date"].isin(dates[train_end:val_end - GAP])]
test = data[data["date"].isin(dates[val_end:])]
train_val = pd.concat([train, val])

for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:10s} {part['date'].min():%Y-%m-%d} to {part['date'].max():%Y-%m-%d}   "
          f"{len(part):5,} rows   {part['target'].mean():.1%} up days")

In [ ]:
MODELS = {
    "logistic_regression": (LogisticRegression(max_iter=2000),
                            {"C": [0.01, 0.1, 1, 10], "class_weight": [None, "balanced"]}),
    "decision_tree": (DecisionTreeClassifier(random_state=RANDOM_STATE),
                      {"max_depth": [2, 3, 5, 8], "min_samples_leaf": [20, 50, 100], "class_weight": [None, "balanced"]}),
    "random_forest": (RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE),
                      {"max_depth": [3, 5, 8], "min_samples_leaf": [20, 50], "max_features": ["sqrt", 0.5],
                       "class_weight": [None, "balanced"]}),
    "svm": (SVC(kernel="rbf"),
            {"C": [0.1, 1, 10], "gamma": ["scale", 0.01], "class_weight": [None, "balanced"]}),
}
FEATURE_SETS = {"market_only": MARKET_FEATURES, "market_plus_sentiment": MARKET_FEATURES + SENTIMENT_FEATURES}


def evaluate(model, X, y):
    pred = model.predict(X)
    score = model.decision_function(X) if hasattr(model, "decision_function") else model.predict_proba(X)[:, 1]
    return {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0), "f1_macro": f1_score(y, pred, average="macro", zero_division=0),
            "roc_auc": roc_auc_score(y, score)}


results, confusions, final_models = [], {}, {}

baseline = DummyClassifier(strategy="most_frequent").fit(train_val[MARKET_FEATURES], train_val["target"])
base_pred = baseline.predict(test[MARKET_FEATURES])
results.append({"features": "baseline", "model": "majority_class",
                "accuracy": accuracy_score(test["target"], base_pred),
                "balanced_accuracy": balanced_accuracy_score(test["target"], base_pred),
                "precision": precision_score(test["target"], base_pred, zero_division=0),
                "recall": recall_score(test["target"], base_pred, zero_division=0),
                "f1": f1_score(test["target"], base_pred, zero_division=0),
                "f1_macro": f1_score(test["target"], base_pred, average="macro", zero_division=0), "roc_auc": 0.5})
confusions[("baseline", "majority_class")] = confusion_matrix(test["target"], base_pred, labels=[0, 1])

for set_name, columns in FEATURE_SETS.items():
    for model_name, (model, grid) in MODELS.items():
        # 1) try every setting, keep the best one on validation
        best_settings, best_val = None, -1
        for values in itertools.product(*grid.values()):
            settings = dict(zip(grid.keys(), values))
            candidate = Pipeline([("scale", StandardScaler()), ("clf", clone(model).set_params(**settings))])
            candidate.fit(train[columns], train["target"])
            val_f1 = f1_score(val["target"], candidate.predict(val[columns]), average="macro")
            if val_f1 > best_val:
                best_settings, best_val = settings, val_f1

        # 2) retrain on train + validation, 3) evaluate once on test
        final = Pipeline([("scale", StandardScaler()), ("clf", clone(model).set_params(**best_settings))])
        final.fit(train_val[columns], train_val["target"])
        final_models[(set_name, model_name)] = final
        confusions[(set_name, model_name)] = confusion_matrix(test["target"], final.predict(test[columns]), labels=[0, 1])
        results.append({"features": set_name, "model": model_name, "best_settings": best_settings,
                        "validation_f1_macro": best_val, **evaluate(final, test[columns], test["target"])})
        print(f"done: {set_name:22s} {model_name}")

results = pd.DataFrame(results)
results.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)

## 5. Results

### 5.1 Test-set performance

In [ ]:
results.drop(columns=["best_settings"])

In [ ]:
comparison = results[results["features"] != "baseline"].pivot(index="model", columns="features",
                                                                values=["accuracy", "balanced_accuracy", "f1_macro", "roc_auc"])
for metric in ["accuracy", "balanced_accuracy", "f1_macro", "roc_auc"]:
    comparison[(metric, "difference")] = comparison[(metric, "market_plus_sentiment")] - comparison[(metric, "market_only")]
comparison = comparison.sort_index(axis=1)
comparison.to_csv(OUTPUT_DIR / "market_vs_sentiment.csv")
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_data = results[results["features"] != "baseline"]
for ax, metric in zip(axes, ["accuracy", "roc_auc"]):
    sns.barplot(data=plot_data, x="model", y=metric, hue="features", ax=ax)
    ax.axhline(results.loc[results["features"] == "baseline", metric].iloc[0], ls="--", c="grey", label="baseline")
    ax.set_ylim(0.4, max(0.7, plot_data[metric].max() + 0.05))
    ax.set_title(f"Test {metric}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=130)
plt.show()

### 5.2 Confusion matrices

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for ax, (set_name, model_name) in zip(axes.flat, confusions):
    sns.heatmap(confusions[(set_name, model_name)], annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["down", "up"], yticklabels=["down", "up"])
    ax.set_title(f"{model_name}\n({set_name})", fontsize=9)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=130)
plt.show()

### 5.3 Which features matter? (random forest)

In [ ]:
forest = final_models[("market_plus_sentiment", "random_forest")].named_steps["clf"]
importance = pd.Series(forest.feature_importances_, index=MARKET_FEATURES + SENTIMENT_FEATURES).sort_values()
colors = ["tab:orange" if f in SENTIMENT_FEATURES else "tab:blue" for f in importance.index]
importance.plot.barh(figsize=(7, 7), color=colors, title="Random forest feature importance (orange = sentiment)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=130)
plt.show()

### 5.4 Exploratory k-means clustering

Without using the label, we group stock-days by their market and news behaviour, then check whether some clusters are followed by noticeably more up days than others. We choose the number of clusters *k* with the silhouette score and compare clusters to the label with a cross-tabulation. Clustering is fit on the training period only.

In [ ]:
CLUSTER_FEATURES = ["ret_1", "vol_5", "volume_chg", "sent_net_mean", "news_count_log"]
X_cluster = StandardScaler().fit_transform(train[CLUSTER_FEATURES])

silhouettes = {}
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_cluster)
    silhouettes[k] = silhouette_score(X_cluster, labels, sample_size=min(3000, len(X_cluster)), random_state=RANDOM_STATE)
best_k = max(silhouettes, key=silhouettes.get)
print("Silhouette score by k:", {k: round(s, 3) for k, s in silhouettes.items()}, "-> best k =", best_k)

clustered = train.assign(cluster=KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_cluster))
profile = clustered.groupby("cluster")[CLUSTER_FEATURES + ["target"]].mean().rename(columns={"target": "share_up_next_day"})
profile.insert(0, "size", clustered["cluster"].value_counts().sort_index())
profile.to_csv(OUTPUT_DIR / "kmeans_profile.csv")
display(profile)
pd.crosstab(clustered["cluster"], clustered["target"].map({0: "down", 1: "up"}), normalize="index")

## 6. Discussion and limitations

*Fill this in after running on the real data. Questions to answer:*

* Did adding sentiment improve test performance for most models, or only one? By how much compared to the majority baseline?
* Is the improvement bigger than what we'd expect from noise? (With a few hundred test days, a 1–2 point accuracy difference can easily be chance.)
* Which sentiment features did the random forest rely on?
* Did any k-means cluster have a clearly different share of up days?

**Limitations we already know about:**

* **News coverage and matching:** GDELT only gives headlines (not full articles) and some company mentions are missed or wrong even after filtering.
* **Timing:** GDELT's timestamp is when GDELT *saw* the article, which can be later than when it was published.
* **Noisy sentiment:** FinBERT scores a headline in isolation and doesn't know whether the news was already expected by the market.
* **Small universe and period:** five large tech stocks over three years; results may not generalize to other stocks or market conditions.
* **Daily resolution:** a lot of news is priced in within minutes, which daily data can't capture.
* **Transaction costs** are ignored. This is a classification study, not a trading strategy.

## 7. Conclusion and future work

*Fill this in after running on the real data:* answer the research question in one or two sentences, based on Section 5.

**Possible future work:** use full article text instead of headlines; add more stocks and sectors; use intraday prices; try models built for sequences (e.g. LSTMs); and use walk-forward (rolling) evaluation to check that results hold up across different market periods.

## 8. References and dataset citations

* **GDELT Project.** GDELT DOC 2.0 API. https://www.gdeltproject.org/ — Leetaru, K. & Schrodt, P. (2013). *GDELT: Global data on events, location, and tone, 1979–2012.* ISA Annual Convention. Terms of use: https://www.gdeltproject.org/about.html#termsofuse
* **Stooq.** Historical daily stock quotes. https://stooq.com/ (check terms of use on the site before use).
* **FinBERT:** Araci, D. (2019). *FinBERT: Financial Sentiment Analysis with Pre-trained Language Models.* arXiv:1908.10063. Model: https://huggingface.co/ProsusAI/finbert
* Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12, 2825–2830.
* Wolf, T. et al. (2020). *Transformers: State-of-the-Art Natural Language Processing.* EMNLP: System Demonstrations.

In [ ]:
# Save the final modeling table so the results can be reproduced without downloading again
data.to_csv(OUTPUT_DIR / "model_dataset.csv", index=False)
print("Saved files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))